# Get to Know a Dataset: NovoMCP Open Corpus (lite)

**Registry of Open Data landing page:** https://registry.opendata.aws/novomcp-open-corpus-lite/  
**Also available:** [Kaggle](https://www.kaggle.com/datasets/novomcp/novomcp-corpus-lite-122m) · [Zenodo (DOI 10.5281/zenodo.21710894)](https://doi.org/10.5281/zenodo.21710894)  
**License:** CC-BY-4.0

The NovoMCP Open Corpus (lite) is a precomputed, analysis-ready snapshot of the full PubChem small-molecule space — one row per compound (122,454,458 rows), keyed by PubChem CID. Each row carries RDKit physicochemical descriptors, drug-likeness (QED) and synthetic-accessibility scores, ~30 ADMET/toxicity predictions from NovoMCP's own models (trained on the public Therapeutics Data Commons benchmark), and PAINS structural-alert flags. It exists so researchers don't have to recompute the same descriptors and property predictions across the whole chemical space — filter, similarity-search, and build ML training sets against a shared, consistent substrate.

> The ADMET fields are model *estimates* for screening and triage, not experimental measurements. Use them to rank and prioritize, not to conclude.

In [ ]:
# The public, program-sponsored bucket. Reads are anonymous (no AWS credentials needed).
BUCKET = "novomcp-open-corpus"          # sponsored-account S3 bucket
PREFIX = "novomcp-open-corpus-lite/"    # flat prefix of the parquet shards
REGION = "us-east-2"

import boto3
from botocore import UNSIGNED
from botocore.config import Config
import polars as pl
import matplotlib.pyplot as plt

## Q: How have you organized your dataset? Help us understand the key-prefix structure of your S3 bucket.

The corpus is a single flat prefix of **1,409 Apache Parquet shards** (`pubchem_novomcp_lite_00000.parquet` … `pubchem_novomcp_lite_01408.parquet`), sharded by PubChem CID range and molecular-weight band. Every shard shares the **same 59-column schema**, so any shard is representative and the whole set concatenates cleanly. There is no nested hierarchy — one prefix, one file type. Below we list the first few objects.

In [ ]:
s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=5)
for obj in resp.get("Contents", []):
    print(f"{obj['Key']:<48} {obj['Size']/1e6:6.1f} MB")
print("...")
total = s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX)
n = sum(len(p.get("Contents", [])) for p in total)
print(f"{n} parquet shards under s3://{BUCKET}/{PREFIX}")

## Q: What data formats are present in your dataset? What kinds of data are stored using these formats?

**Apache Parquet (Snappy compression).** Parquet is columnar, so cloud query engines read only the columns and row-groups a query touches (column + predicate pushdown) — you can answer a question over 122M rows while scanning a fraction of the 27 GB. Sharding by CID/MW band enables parallel scans and partition pruning.

**Recommended tooling:** [Polars](https://pola.rs), [DuckDB](https://duckdb.org), PyArrow, or pandas for local/interactive work; **Amazon Athena** (serverless SQL over the S3 Parquet), **AWS Glue**, or **EMR/Spark** for at-scale querying without moving the data. Because reads are anonymous, none of these require AWS credentials to *read* the public bucket.

## Q: Can you show us an example of downloading and loading data from your dataset?

Loading a single shard with Polars (anonymous S3 read) and inspecting the schema:

In [ ]:
uri = f"s3://{BUCKET}/{PREFIX}pubchem_novomcp_lite_00000.parquet"
df = pl.read_parquet(uri, storage_options={"aws_region": REGION, "aws_skip_signature": "true"})
print(f"shard shape: {df.shape}")
print(f"columns ({len(df.columns)}): {df.columns}")
df.select(["cid", "smiles", "molecular_weight", "xlogp", "tpsa", "qed",
           "hepatotoxicity_probability", "cardiotoxicity_max_probability", "has_pains"]).head()

## Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset.

The distribution of drug-likeness (QED) across one shard — a quick sense of where PubChem sits in property space.

In [ ]:
sample = pl.read_parquet(uri, columns=["molecular_weight", "qed"],
                         storage_options={"aws_region": REGION, "aws_skip_signature": "true"})
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(sample["qed"], bins=50, color="#4C72B0")
ax[0].set_xlabel("QED (drug-likeness)"); ax[0].set_ylabel("compounds"); ax[0].set_title("Drug-likeness")
ax[1].hist(sample["molecular_weight"].filter(sample["molecular_weight"] < 1000), bins=50, color="#55A868")
ax[1].set_xlabel("Molecular weight (g/mol)"); ax[1].set_title("Molecular weight (< 1000)")
plt.tight_layout(); plt.show()

## Q: What is one question that you have answered using these data?

**How much of PubChem passes a simple oral-drug-like screen — and how many of those *also* look clean on predicted cardiotoxicity and PAINS?** Because every property is precomputed, this is one SQL scan over the whole corpus rather than a 122M-molecule RDKit + inference job. We run it in-place with DuckDB (`httpfs` reads the public Parquet directly).

In [ ]:
import duckdb
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"SET s3_region='{REGION}'; SET s3_use_ssl=true;")
glob = f"s3://{BUCKET}/{PREFIX}*.parquet"
con.sql(f"""
SELECT
  count(*)                                                           AS total,
  count(*) FILTER (molecular_weight < 500 AND hbd <= 5 AND hba <= 10
                   AND xlogp < 5)                                    AS lipinski_ok,
  count(*) FILTER (molecular_weight < 500 AND hbd <= 5 AND hba <= 10
                   AND xlogp < 5 AND qed > 0.5)                      AS druglike,
  count(*) FILTER (molecular_weight < 500 AND hbd <= 5 AND hba <= 10
                   AND xlogp < 5 AND qed > 0.5
                   AND cardiotoxicity_max_probability < 0.5
                   AND NOT has_pains)                                AS druglike_clean
FROM read_parquet('{glob}')
""").show()

## Q: What is one unanswered question that you think could be answered using these data? (Community challenge)

**Challenge — chemotype discovery under a target property profile.** Using only the precomputed columns, can you identify *novel, synthetically-accessible* chemotypes (low `synthetic_accessibility`, high `qed`) that sit in an underexplored corner of property space while staying clean on the predicted-toxicity panel? Concretely: cluster the drug-like, low-alert subset by scaffold, then surface scaffolds that are (a) rare in the current corpus, (b) favorable across the ADMET vector, and (c) not already common in approved-drug space.

A second, complementary challenge for the ML community: **treat the ~30 ADMET/toxicity columns as weak labels** and benchmark a new multi-task ADMET model against them at PubChem scale — where does a model trained on the public TDC benchmark agree or diverge with these predictions across the full chemical space, and what does that reveal about applicability domain?

**Recommendations for researchers:** start from a single shard to prototype, then scale the same Polars/DuckDB query to the full `*.parquet` glob (or Amazon Athena) once your logic is set. The uniform 59-column schema means nothing changes between one shard and all 1,409. Remember the ADMET fields are screening-grade predictions — use them to *rank and shortlist*, then validate the shortlist with higher-fidelity methods or experiment.